In [ ]:
import torch                                              # PyTorch main package
import torch.nn as nn                                     # Neural network layers
from torch.utils.data import DataLoader                   # Mini-batch loading
from torchvision import datasets, transforms, models       # MNIST + transforms + MobileNetV3
from tqdm import tqdm                                     # Progress bar

ROOT = "."                                                # Folder that contains ./MNIST/raw and ./MNIST/processed
BS = 128                                                  # Batch size
WEIGHTS_PATH = "mobilenetv3_mnist.pth"                    # Saved weights file name
device = "cuda" if torch.cuda.is_available() else "cpu"    # GPU if available else CPU

tfm_test = transforms.Compose([                           # Same test transforms as training
    transforms.Resize(224),                               # Resize to ImageNet-like size
    transforms.Grayscale(num_output_channels=3),           # 1-channel -> 3-channel
    transforms.ToTensor(),                                 # PIL -> Tensor
    transforms.Normalize([0.485,0.456,0.406],              # ImageNet mean
                         [0.229,0.224,0.225]),             # ImageNet std
])

test_ds = datasets.MNIST(root=ROOT,                       # Load MNIST test split
                         train=False,                     # Test set
                         download=True,                   # Ensure processed exists
                         transform=tfm_test)              # Apply transforms

test_dl = DataLoader(test_ds,                             # DataLoader for test
                     batch_size=BS,                       # Batch size
                     shuffle=False,                       # No shuffling for evaluation
                     num_workers=2,                       # Workers (set 0 if Windows issues)
                     pin_memory=True)                     # Faster CPU->GPU copies

m = models.mobilenet_v3_large(pretrained=False)           # Create MobileNetV3-Large (no need to download ImageNet weights)
m.classifier[-1] = nn.Linear(m.classifier[-1].in_features,# Ensure final layer matches MNIST classes
                             10)                          # 10 digits
m.load_state_dict(torch.load(WEIGHTS_PATH, map_location=device))  # Load your trained weights
m = m.to(device)                                          # Move model to device
m.eval()                                                  # Eval mode

correct = 0                                               # Correct predictions counter
total = 0                                                 # Total samples counter

with torch.no_grad():                                     # No gradients needed for inference
    for x, y in tqdm(test_dl, desc="Test"):               # Loop over test batches
        x = x.to(device)                                  # Move images to device
        y = y.to(device)                                  # Move labels to device
        out = m(x)                                        # Forward pass -> logits
        pred = out.argmax(1)                              # Predicted class
        correct += (pred == y).sum().item()               # Add correct count
        total += y.numel()                                 # Add batch size

print("Test Accuracy:", correct / total)                  # Print final accuracy